In [1]:
import os
from dotenv import load_dotenv

import requests
from langchain_core.tools import tool

@tool
def get_current_time(city: str) -> str:
    """Get the current local time for a city in the world."""

    url = "https://timezones.live/api/v1/time"

    params = {
        "city": city
    }

    response = requests.get(url, params=params)

    response.raise_for_status()

    data = response.json()["data"]

    return (
        f"Current time in {data['city']}, "
        f"{data['country']} is {data['currentTime']} "
        f"on {data['currentDate']}."
    )

print(type(get_current_time))

print("\nName:")
print(get_current_time.name)

print("\nDescription:")
print(get_current_time.description)

print("\nArguments:")
print(get_current_time.args)

<class 'langchain_core.tools.structured.StructuredTool'>

Name:
get_current_time

Description:
Get the current local time for a city in the world.

Arguments:
{'city': {'title': 'City', 'type': 'string'}}


In [2]:
result = get_current_time.invoke({
    "city": "Tokyo"
})

print(result)

Current time in Tokyo, Japan is 03:10 PM on Mon, Sep 14.


In [3]:
from langchain_groq import ChatGroq

load_dotenv()

llm = ChatGroq(
    model=os.environ.get("GROQ_MODEL_OPEN"),
    temperature=0
)

llm_with_tools = llm.bind_tools(
    [get_current_time]
)

In [4]:
user_query = "What time is it currently in Tokyo?"
user_query2 = "What is 10 + 20?"

response = llm_with_tools.invoke(user_query)

print("CONTENT:")
print(response.content)

print("\nTOOL CALLS:")
print(response.tool_calls)

CONTENT:


TOOL CALLS:
[{'name': 'get_current_time', 'args': {'city': 'Tokyo'}, 'id': 'fc_6b7842af-8f44-482f-a605-c812fd741be9', 'type': 'tool_call'}]


In [6]:
tool_call = response.tool_calls[0]

tool_result = get_current_time.invoke(
    tool_call["args"]
)

print("TOOL RESULT:")
print(tool_result)

TOOL RESULT:
Current time in Tokyo, Japan is 03:10 PM on Mon, Sep 14.


In [7]:
from langchain_core.messages import ToolMessage

tool_message = ToolMessage(
    content=str(tool_result),
    tool_call_id=tool_call["id"]
)

messages = [
    {
        "role": "user",
        "content": user_query
    },
    response,
    tool_message
]

final_response = llm_with_tools.invoke(
    messages,
    tool_choice="none"
)

print("FINAL ANSWER:")
print(final_response.content)

FINAL ANSWER:
It’s currently **3:10 PM on Monday, September 14** in Tokyo.
